# Prediction under sensor loss (simulation)

Hold out one quarter of participants. Train regularized logistic classifiers on intact measurements or on intact and masked copies. Test both models with all features and with the second sensor group replaced by the training mean. Compare AUROC under sensor loss. Generate paired measurements of shared latent signals for 32 subjects. The two sensor groups have different noise levels and subject-specific offsets.

In [ ]:
from pathlib import Path
import subprocess, sys
root = Path.cwd() / 'missing-eeg-sensors-heavy-run'
root.mkdir(exist_ok=True)
subprocess.run([sys.executable,'-m','pip','install',*['numpy==2.2.6', 'scipy==1.15.3', 'scikit-learn==1.7.2', 'joblib==1.6.0']],check=True)

In [ ]:
(root / 'train.py').write_text('import json\nimport os\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import (\n    average_precision_score,\n    roc_auc_score,\n)\n\nROOT = Path(__file__).resolve().parent\nDATA = Path(os.environ.get("PORTFOLIO_DATA", ROOT / "data"))\nSEEDS = [13, 37, 61]\n\n\ndef save(obj, name):\n    folder = ROOT / "checkpoints"\n    folder.mkdir(exist_ok=True)\n    joblib.dump(obj, folder / (name + ".joblib"), compress=3)\n\n\ndef scores(y, p):\n    return {\n        "auroc": float(roc_auc_score(y, p)),\n        "average_precision": float(average_precision_score(y, p)),\n    }\n\n\ndef dataset(seed):\n    rng = np.random.default_rng(seed)\n    x = []\n    y = []\n    g = []\n    for p in range(32):\n        latent = rng.normal(size=(130, 3))\n        target = latent[:, 0] + 0.7 * latent[:, 1] + rng.normal(0, 0.7, 130) > 0\n        views = np.c_[\n            latent + rng.normal(0, 0.9, (130, 3)), latent + rng.normal(0, 0.5, (130, 3))\n        ]\n        x.extend(views + rng.normal(0, 0.45, 6))\n        y.extend(target)\n        g.extend([p] * 130)\n    return np.array(x), np.array(y), np.array(g)\n\n\nfrom sklearn.model_selection import GroupShuffleSplit\n\n\ndef run(seed):\n    x, y, g = dataset(seed)\n    tr, te = next(\n        GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=seed).split(x, y, g)\n    )\n    mean = x[tr].mean(0)\n    std = x[tr].std(0) + 1e-6\n    a = (x[tr] - mean) / std\n    b = (x[te] - mean) / std\n\n    def missing(z):\n        z = z.copy()\n        z[:, z.shape[1] // 2 :] = 0\n        return z\n\n    out = {}\n    for name in ["standard", "dropout"]:\n        f = a\n        target = y[tr]\n        if name == "dropout":\n            f = np.concatenate([a, missing(a)])\n            target = np.tile(target, 2)\n        model = LogisticRegression(C=0.08, max_iter=500, class_weight="balanced").fit(\n            f, target\n        )\n        for cond, z in [("intact", b), ("missing", missing(b))]:\n            out[name + "_" + cond] = scores(y[te], model.predict_proba(z)[:, 1])\n        save((mean, std, model, np.unique(g[te])), f"{name}-{seed}")\n    return out\n\n\nif __name__ == "__main__":\n    runs = {s: run(s) for s in SEEDS}\n    first = next(iter(runs.values()))\n    means = {\n        k: {m: float(np.mean([r[k][m] for r in runs.values()])) for m in v}\n        for k, v in first.items()\n    }\n    (ROOT / "results.json").write_text(\n        json.dumps({"seeds": SEEDS, "runs": runs, "means": means}, indent=2)\n    )\n    print(means)\n',encoding='utf8')
subprocess.run([sys.executable,'train.py'],cwd=root,check=True)

In [ ]:
print((root/'results.json').read_text())